# softmax回归的简洁实现
:label:`sec_softmax_concise`

在 :numref:`sec_linear_concise`中，
我们发现(**通过深度学习框架的高级API能够使实现**)
(~~softmax~~)
线性(**回归变得更加容易**)。
同样，通过深度学习框架的高级API也能更方便地实现softmax回归模型。
本节如在 :numref:`sec_softmax_scratch`中一样，
继续使用Fashion-MNIST数据集，并保持批量大小为256。


In [8]:
import torch  # PyTorch深度学习框架
from torch import nn  # 神经网络模块
from d2l import torch as d2l  # 动手学深度学习工具包

In [9]:
batch_size = 256  # 每批次读取256个样本

# 加载Fashion-MNIST训练集和测试集
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

## 初始化模型参数

如我们在 :numref:`sec_softmax`所述，
[**softmax回归的输出层是一个全连接层**]。
因此，为了实现我们的模型，
我们只需在`Sequential`中添加一个带有10个输出的全连接层。
同样，在这里`Sequential`并不是必要的，
但它是实现深度模型的基础。
我们仍然以均值0和标准差0.01随机初始化权重。


In [10]:
# 构建Softmax回归模型
net = nn.Sequential(
    nn.Flatten(),       # 将28×28图像展平为784维向量
    nn.Linear(784, 10)  # 全连接层：784个输入特征，10个类别输出
)

def init_weights(m):
    """初始化模型权重"""
    if type(m) == nn.Linear:  # 判断是否为全连接层
        # 使用均值为0、标准差为0.01的正态分布初始化权重
        nn.init.normal_(m.weight, std=0.01)

net.apply(init_weights)  # 对模型各层应用权重初始化函数

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=10, bias=True)
)

## 重新审视Softmax的实现
:label:`subsec_softmax-implementation-revisited`

在前面 :numref:`sec_softmax_scratch`的例子中，
我们计算了模型的输出，然后将此输出送入交叉熵损失。
从数学上讲，这是一件完全合理的事情。
然而，从计算角度来看，指数可能会造成数值稳定性问题。

回想一下，softmax函数$\hat y_j = \frac{\exp(o_j)}{\sum_k \exp(o_k)}$，
其中$\hat y_j$是预测的概率分布。
$o_j$是未规范化的预测$\mathbf{o}$的第$j$个元素。
如果$o_k$中的一些数值非常大，
那么$\exp(o_k)$可能大于数据类型容许的最大数字，即*上溢*（overflow）。
这将使分母或分子变为`inf`（无穷大），
最后得到的是0、`inf`或`nan`（不是数字）的$\hat y_j$。
在这些情况下，我们无法得到一个明确定义的交叉熵值。

解决这个问题的一个技巧是：
在继续softmax计算之前，先从所有$o_k$中减去$\max(o_k)$。
这里可以看到每个$o_k$按常数进行的移动不会改变softmax的返回值：

$$
\begin{aligned}
\hat y_j & =  \frac{\exp(o_j - \max(o_k))\exp(\max(o_k))}{\sum_k \exp(o_k - \max(o_k))\exp(\max(o_k))} \\
& = \frac{\exp(o_j - \max(o_k))}{\sum_k \exp(o_k - \max(o_k))}.
\end{aligned}
$$


在减法和规范化步骤之后，可能有些$o_j - \max(o_k)$具有较大的负值。
由于精度受限，$\exp(o_j - \max(o_k))$将有接近零的值，即*下溢*（underflow）。
这些值可能会四舍五入为零，使$\hat y_j$为零，
并且使得$\log(\hat y_j)$的值为`-inf`。
反向传播几步后，我们可能会发现自己面对一屏幕可怕的`nan`结果。

尽管我们要计算指数函数，但我们最终在计算交叉熵损失时会取它们的对数。
通过将softmax和交叉熵结合在一起，可以避免反向传播过程中可能会困扰我们的数值稳定性问题。
如下面的等式所示，我们避免计算$\exp(o_j - \max(o_k))$，
而可以直接使用$o_j - \max(o_k)$，因为$\log(\exp(\cdot))$被抵消了。

$$
\begin{aligned}
\log{(\hat y_j)} & = \log\left( \frac{\exp(o_j - \max(o_k))}{\sum_k \exp(o_k - \max(o_k))}\right) \\
& = \log{(\exp(o_j - \max(o_k)))}-\log{\left( \sum_k \exp(o_k - \max(o_k)) \right)} \\
& = o_j - \max(o_k) -\log{\left( \sum_k \exp(o_k - \max(o_k)) \right)}.
\end{aligned}
$$

我们也希望保留传统的softmax函数，以备我们需要评估通过模型输出的概率。
但是，我们没有将softmax概率传递到损失函数中，
而是[**在交叉熵损失函数中传递未规范化的预测，并同时计算softmax及其对数**]，
这是一种类似["LogSumExp技巧"](https://en.wikipedia.org/wiki/LogSumExp)的聪明方式。


In [11]:
# 定义交叉熵损失函数
# reduction='none'：返回每个样本的损失，不进行求和或平均
loss = nn.CrossEntropyLoss(reduction='none')

## 优化算法

在这里，我们(**使用学习率为0.1的小批量随机梯度下降作为优化算法**)。
这与我们在线性回归例子中的相同，这说明了优化器的普适性。


In [12]:
# 使用小批量随机梯度下降(SGD)优化模型参数
# net.parameters()：获取模型所有可训练参数
# lr=0.1：设置学习率为0.1
trainer = torch.optim.SGD(net.parameters(), lr=0.1)

## 训练

接下来我们[**调用**] :numref:`sec_softmax_scratch`中(~~之前~~)
(**定义的训练函数来训练模型**)。


In [14]:
import d2l
from d2l import torch as d2l_torch

print("D2L版本：", d2l.__version__)
print("是否存在train_ch3：", hasattr(d2l_torch, "train_ch3"))
print("D2L路径：", d2l_torch.__file__)

D2L版本： 1.0.3
是否存在train_ch3： False
D2L路径： /root/miniconda3/envs/d2l-zh/lib/python3.8/site-packages/d2l/torch.py


In [15]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, trainer):
    """训练分类模型"""
    for epoch in range(num_epochs):
        net.train()  # 训练模式
        train_loss, train_acc, n = 0.0, 0.0, 0

        for X, y in train_iter:
            y_hat = net(X)  # 前向传播
            l = loss(y_hat, y)  # 计算损失

            trainer.zero_grad()  # 清空梯度
            l.mean().backward()  # 反向传播
            trainer.step()  # 更新参数

            train_loss += l.sum().item()
            train_acc += (y_hat.argmax(dim=1) == y).sum().item()
            n += y.numel()

        net.eval()  # 评估模式
        test_acc, test_n = 0, 0

        with torch.no_grad():
            for X, y in test_iter:
                test_acc += (net(X).argmax(dim=1) == y).sum().item()
                test_n += y.numel()

        print(f"epoch {epoch + 1}: "
              f"loss {train_loss / n:.4f}, "
              f"train acc {train_acc / n:.4f}, "
              f"test acc {test_acc / test_n:.4f}")

In [17]:
num_epochs = 10  # 训练轮数，完整遍历训练集10次

# 调用D2L封装的训练函数
train_ch3(
    net,          # Softmax回归模型
    train_iter,   # 训练集迭代器
    test_iter,    # 测试集迭代器
    loss,         # 交叉熵损失函数
    num_epochs,   # 训练轮数
    trainer       # SGD优化器
)

epoch 1: loss 0.7865, train acc 0.7509, test acc 0.7356
epoch 2: loss 0.5716, train acc 0.8134, test acc 0.8082
epoch 3: loss 0.5247, train acc 0.8261, test acc 0.8206
epoch 4: loss 0.5013, train acc 0.8324, test acc 0.8173
epoch 5: loss 0.4854, train acc 0.8371, test acc 0.8274
epoch 6: loss 0.4738, train acc 0.8415, test acc 0.8230
epoch 7: loss 0.4652, train acc 0.8435, test acc 0.8258
epoch 8: loss 0.4571, train acc 0.8457, test acc 0.8207
epoch 9: loss 0.4523, train acc 0.8468, test acc 0.8243
epoch 10: loss 0.4471, train acc 0.8489, test acc 0.8283


和以前一样，这个算法使结果收敛到一个相当高的精度，而且这次的代码比之前更精简了。

## 小结

* 使用深度学习框架的高级API，我们可以更简洁地实现softmax回归。
* 从计算的角度来看，实现softmax回归比较复杂。在许多情况下，深度学习框架在这些著名的技巧之外采取了额外的预防措施，来确保数值的稳定性。这使我们避免了在实践中从零开始编写模型时可能遇到的陷阱。

[Discussions](https://discuss.d2l.ai/t/1793)
